                  ABORDAGEM 1
SMS → Normalização → TF-IDF → 24 features → Modelos
                                      ↓
                                  Métricas
                                     
                  ABORDAGEM 2
SMS → Normalização → EMBEDDINGS → 24 features → Modelos
                                      ↓
                                  Métricas

Os embeddings conseguem melhorar o desempenho do classificador em relação ao TF-IDF para este dataset de SMS?

Carregar o dataset.
Separar Mensagem e Possivel golpe.
Separar as 24 features estruturadas.
Fazer o train_test_split exatamente como antes.
Normalizar as mensagens.
Gerar os embeddings somente usando os dados de treinamento para evitar vazamento.
Combinar embeddings + features estruturadas.
Salvar tudo em data/processed.

Para o experimento, vamos usar o modelo sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2. Ele é adequado para nosso caso porque é multilíngue, suporta português e transforma cada mensagem em um vetor denso de 384 dimensões.

Instalar as bibliotecas

In [1]:
# Importa o módulo sys para identificar o Python utilizado pelo notebook.
import sys

# Instala a biblioteca Sentence Transformers no mesmo ambiente Python do notebook.
!{sys.executable} -m pip install -U sentence-transformers

In [2]:
# Importa a biblioteca Pandas para manipulação dos dados.
import pandas as pd

# Importa a biblioteca NumPy para operações numéricas.
import numpy as np

# Importa a biblioteca Joblib para salvar os dados preparados.
import joblib

# Importa a biblioteca re para trabalhar com expressões regulares.
import re

# Importa o modelo SentenceTransformer para geração dos embeddings.
from sentence_transformers import SentenceTransformer

# Importa a função train_test_split para separar treinamento e teste.
from sklearn.model_selection import train_test_split

Carregar o dataset

In [3]:
# Define o caminho relativo para o dataset original.
caminho_dataset = "../data/raw/DATASET - Dataset_limpo_V1.csv"

# Carrega o arquivo CSV em um DataFrame.
df = pd.read_csv(caminho_dataset)

# Exibe as primeiras linhas para confirmar que o dataset foi carregado corretamente.
df.head()

,Mensagem,solicita_senha,solicita_codigo_autenticacao,solicita_dados_bancarios,solicita_dados_pessoais,solicita_dados_cartao,solicita_atualizacao_cadastro,solicita_pagamento,solicita_clique,solicita_ligacao,...,possui_reembolso,possui_vantagem_inesperada,possui_oferta_financeira,possui_url,possui_url_encurtada,possui_telefone,possui_pix,possui_erros_ortografia_gramatica,possui_chamada_acao,possivel_golpe
0,Sabia que da pra curtir tudo da Claro? 350 MEG...,0,0,0,0,0,0,0,1,1,...,0,0,1,1,1,1,0,1,1,0
1,Seu dia a dia pede MUITA INTERNET! Entao vem p...,0,0,0,0,0,0,0,1,0,...,0,0,1,1,1,0,0,1,1,0
2,"Cliente Claro,nao identificamos pgt fat do ser...",0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,1,1,0
3,Claro Multi ta na sua com oferta especial: 600...,0,0,0,0,0,0,0,1,1,...,0,0,1,1,1,1,0,1,1,0
4,Claro Multi ta na sua com oferta especial: 600...,0,0,0,0,0,0,0,1,1,...,0,0,1,1,1,1,0,1,1,0


Definir target e atributos

In [5]:
# Define o nome da coluna que representa a variável que queremos prever.
coluna_alvo = "possivel_golpe"

# Cria a variável y_target contendo somente a variável que queremos prever.
y_target = df[coluna_alvo]

# Cria o DataFrame X_atributos contendo todas as características utilizadas pelo modelo.
X_atributos = df.drop(columns=[coluna_alvo])

Separar as mensagens

In [6]:
# Cria uma variável contendo somente o texto original das mensagens SMS.
mensagens = X_atributos["Mensagem"]

# Remove a coluna de texto dos atributos estruturados.
X_atributos = X_atributos.drop(columns=["Mensagem"])

mensagens
    ↓
texto dos SMS

X_atributos
    ↓
24 características estruturadas

Transformar carga emocional em número

In [7]:
# Cria um dicionário que associa cada nível de carga emocional a um valor numérico.
mapa_carga_emocional = {
    "Baixo": 0,
    "Médio": 1,
    "Alto": 2
}

# Converte os níveis de carga emocional em valores numéricos.
X_atributos["carga_emocional"] = X_atributos["carga_emocional"].map(mapa_carga_emocional)

Dividir treinamento e teste

In [8]:
# Divide as mensagens em conjuntos de treinamento e teste mantendo a proporção das classes.
mensagens_treino, mensagens_teste, y_treino, y_teste = train_test_split(
    mensagens,
    y_target,
    test_size=0.20,
    random_state=42,
    stratify=y_target
)

# Divide os atributos estruturados utilizando exatamente a mesma estratégia de separação.
X_atributos_treino, X_atributos_teste, _, _ = train_test_split(
    X_atributos,
    y_target,
    test_size=0.20,
    random_state=42,
    stratify=y_target
)

Normalizar as mensagens

In [9]:
# Define uma função responsável pela normalização básica das mensagens.
def normalizar_mensagem(mensagem):

    # Verifica se a mensagem está ausente.
    if pd.isna(mensagem):

        # Retorna uma string vazia quando a mensagem estiver ausente.
        return ""

    # Converte o conteúdo recebido para texto.
    mensagem = str(mensagem)

    # Converte todas as letras da mensagem para minúsculas.
    mensagem = mensagem.lower()

    # Substitui múltiplos espaços, quebras de linha e tabulações por um único espaço.
    mensagem = re.sub(r"\s+", " ", mensagem)

    # Remove espaços desnecessários no início e no final da mensagem.
    mensagem = mensagem.strip()

    # Retorna a mensagem normalizada.
    return mensagem

In [10]:
# Normaliza todas as mensagens utilizadas no treinamento.
mensagens_treino_normalizadas = mensagens_treino.apply(normalizar_mensagem)

# Normaliza todas as mensagens utilizadas no teste.
mensagens_teste_normalizadas = mensagens_teste.apply(normalizar_mensagem)

Carregar o modelo de embeddings

In [12]:
# Define o modelo multilíngue de embeddings que será utilizado no projeto.
nome_modelo_embedding = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Carrega o modelo de embeddings pré-treinado.
modelo_embedding = SentenceTransformer(nome_modelo_embedding)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gerar os embeddings

In [13]:
# Gera os embeddings das mensagens utilizadas no treinamento.
embeddings_treino = modelo_embedding.encode(
    mensagens_treino_normalizadas.tolist(),
    show_progress_bar=True
)

# Gera os embeddings das mensagens utilizadas no teste.
embeddings_teste = modelo_embedding.encode(
    mensagens_teste_normalizadas.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Mensagem
   ↓
Sentence Transformer
   ↓
Vetor denso de 384 dimensões

Verificar os embeddings

In [14]:
# Exibe o formato da matriz de embeddings de treinamento.
print("Formato dos embeddings de treinamento:", embeddings_treino.shape)

# Exibe o formato da matriz de embeddings de teste.
print("Formato dos embeddings de teste:", embeddings_teste.shape)

Formato dos embeddings de treinamento: (1077, 384)
Formato dos embeddings de teste: (270, 384)


| Conjunto    | Mensagens | Dimensões |
| ----------- | --------: | --------: |
| Treinamento |      1077 |       384 |
| Teste       |       270 |       384 |


Combinar embeddings com as features estruturadas

384 dimensões do embedding
+
24 features estruturadas
=
408 características

Converter as features estruturadas.

In [15]:
# Converte os atributos estruturados de treinamento para uma matriz NumPy.
atributos_treino_array = X_atributos_treino.to_numpy()

# Converte os atributos estruturados de teste para uma matriz NumPy.
atributos_teste_array = X_atributos_teste.to_numpy()

Agora fazemos a combinação

In [16]:
# Combina os embeddings das mensagens com as características estruturadas do treinamento.
X_treino_embb = np.hstack([
    embeddings_treino,
    atributos_treino_array
])

# Combina os embeddings das mensagens com as características estruturadas do teste.
X_teste_embb = np.hstack([
    embeddings_teste,
    atributos_teste_array
])

Agora nossa representação final será:

X_treino_embb
        │
        ├── 384 dimensões → Embedding
        │
        └── 24 dimensões  → Features estruturadas
        │
        └── 408 features

Confirmar o tamanho final

In [17]:
# Exibe o formato final dos dados de treinamento.
print("Formato final do treinamento:", X_treino_embb.shape)

# Exibe o formato final dos dados de teste.
print("Formato final do teste:", X_teste_embb.shape)

Formato final do treinamento: (1077, 408)
Formato final do teste: (270, 408)


Salvar os dados preparados

In [18]:
# Cria um dicionário contendo todos os dados necessários para a etapa de modelagem.
dados_preparados_embb = {
    "X_treino_embb": X_treino_embb,
    "X_teste_embb": X_teste_embb,
    "y_treino": y_treino,
    "y_teste": y_teste,
    "modelo_embedding": nome_modelo_embedding
}

# Salva os dados preparados em um arquivo Joblib.
joblib.dump(
    dados_preparados_embb,
    "../data/processed/dados_preparados_embb.joblib"
)

['../data/processed/dados_preparados_embb.joblib']

O que acabamos de construir

                  DATASET
                     │
                     ▼
              Separação dos dados
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
       Mensagem             Features
          │                 estruturadas
          ▼                     │
    Normalização                │
          │                     │
          ▼                     │
 Sentence Transformer           │
          │                     │
          ▼                     │
   384 dimensões                │
          │                     │
          └──────────┬──────────┘
                     ▼
              408 características
                     │
                     ▼
            07_modelagem_embb

Diferentemente do TF-IDF, não precisamos fazer fit_transform() no embedding. O modelo paraphrase-multilingual-MiniLM-L12-v2 já é um modelo pré-treinado; nós apenas usamos encode() para gerar as representações das mensagens. O modelo é especificamente descrito como gerador de embeddings de sentenças/parágrafos de 384 dimensões e multilíngue.